# City profile v3 — capacity tier + SIM/BEP autonomy

Builds the model's two-axis city profile from the two reviewed replacement datasets. The notebook
does not load the retired SINIM extract or reproduce its values. Both axes are consumed unchanged;
only the 0.5 archetype banding is performed here.

- Capacity: `reviews/oef/cl-municipal-capacity-tier/releases/v1`
- Autonomy: `reviews/cl-subdere/cl-subdere-sim-bep/releases/2025`
- Release contract: `review.md`


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "dataset-review").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "dataset-review").exists(), "run from inside the CityCatalyst-global-data repository"

REVIEWS = ROOT / "dataset-review" / "reviews"
HERE = REVIEWS / "oef" / "cl-city-action-fundability" / "releases" / "v3"

tier = pd.read_csv(
    REVIEWS / "oef/cl-municipal-capacity-tier/releases/v1/data/municipal_capacity_tier.csv",
    dtype={"comuna_cut": str})
sim = pd.read_csv(
    REVIEWS / "cl-subdere/cl-subdere-sim-bep/releases/2025/data/sim_municipal_income_2025.csv",
    dtype={"comuna_cut": str, "source_vintage": str})

tier["comuna_cut"] = tier["comuna_cut"].str.zfill(5)
sim["comuna_cut"] = sim["comuna_cut"].str.zfill(5)
print(f"capacity rows {len(tier)} | SIM/BEP rows {len(sim)}")


## Validate the two source contracts

The join is allowed only when both reviews publish the same complete CUT universe. Autonomy is read
from SIM/BEP, while the formula check below verifies that the reviewed output still equals one minus
the recalculated FCM-dependency ratio.


In [ ]:
for name, frame in {"capacity": tier, "SIM/BEP": sim}.items():
    assert len(frame) == 345, f"{name}: expected 345 rows, got {len(frame)}"
    assert frame["comuna_cut"].is_unique, f"{name}: duplicate CUT"
    assert frame["comuna_cut"].str.fullmatch(r"\d{5}").all(), f"{name}: invalid CUT"

capacity_keys = set(tier["comuna_cut"])
autonomy_keys = set(sim["comuna_cut"])
assert capacity_keys == autonomy_keys, {
    "capacity_only": sorted(capacity_keys - autonomy_keys),
    "autonomy_only": sorted(autonomy_keys - capacity_keys),
}

autonomy_check = (1 - sim["fcm_dependency"]).clip(0, 1).round(6)
assert (sim["autonomy"] - autonomy_check).abs().max() <= 1e-6, "SIM/BEP autonomy formula drift"
assert sim["autonomy"].notna().all(), "SIM/BEP autonomy contains nulls"
assert sim["source_vintage"].nunique() == 1, "expected one SIM/BEP source vintage"
print("source assertions passed — exact 345-CUT match")


## Join and band the city archetype

The thresholds and four category names are unchanged from the existing model contract.


In [ ]:
ARCHETYPE = {
    (True, True): "Self-sufficient",
    (False, True): "Delivery-ready",
    (True, False): "Well-resourced",
    (False, False): "Support-ready",
}

profile = tier.merge(
    sim[["comuna_cut", "autonomy", "source_vintage"]],
    on="comuna_cut", how="inner", validate="one_to_one")
profile["city_archetype"] = [
    ARCHETYPE[(autonomy >= 0.5, capacity >= 0.5)]
    for autonomy, capacity in zip(profile["autonomy"], profile["capacity"])]
profile["autonomy_basis"] = "sim_bep_fcm_over_ip_percibido"
profile = profile.rename(columns={"source_vintage": "autonomy_source_vintage"})


## Validate and export

The CSV carries review-level provenance fields for audit. The existing modelled table can continue
selecting only its established columns, so no database schema migration is implied.


In [ ]:
assert len(profile) == 345 and profile["comuna_cut"].is_unique
assert profile[["capacity", "autonomy", "city_archetype"]].notna().all().all()
assert profile["capacity"].between(0, 1).all()
assert profile["autonomy"].between(0, 1).all()
assert set(profile["city_archetype"]) == set(ARCHETYPE.values())

capacity_source = tier.set_index("comuna_cut")["capacity"].sort_index()
capacity_joined = profile.set_index("comuna_cut")["capacity"].sort_index()
autonomy_source = sim.set_index("comuna_cut")["autonomy"].sort_index()
autonomy_joined = profile.set_index("comuna_cut")["autonomy"].sort_index()
assert capacity_joined.equals(capacity_source), "capacity changed during join"
assert autonomy_joined.equals(autonomy_source), "autonomy changed during join"

OUT_COLS = [
    "comuna_cut", "comuna", "population", "tramo", "tramo_label",
    "gl_technical_capacity", "capacity", "autonomy", "city_archetype",
    "capacity_basis", "autonomy_basis", "autonomy_source_vintage", "t4_provisional",
]
out = profile[OUT_COLS].sort_values("comuna_cut").reset_index(drop=True)
out.to_csv(HERE / "data/city_finance_profile.csv", index=False)
print(f"wrote data/city_finance_profile.csv — {out.shape[0]} rows x {out.shape[1]} columns")
print(out["city_archetype"].value_counts().sort_index())


## Result

The output contains 345 complete profiles built only from the capacity-tier and SIM/BEP releases.
Promotion still depends on the four decisions recorded in `review.md`: T4, the T1 evidence gate,
Padre Hurtado and the Natales threshold rule.
